In [ ]:
# Step 1: 仅处理指定bdf文件
import os
import mne
import pandas as pd

bdf_path = r'D:/database/music_eeg/yyh/yyhmusic_260122_3.bdf'
# csv_root = r'D:/database/music_eeg_csv/'
# if not os.path.exists(csv_root):
#     os.makedirs(csv_root)

# 转存为csv（如需）
# raw = mne.io.read_raw_bdf(bdf_path, preload=True, verbose=False)
# data = raw.get_data()
# ch_names = raw.ch_names
# df_bdf = pd.DataFrame(data.T, columns=ch_names)
# csv_path = os.path.join(csv_root, os.path.basename(bdf_path).replace('.bdf', '.csv'))
# df_bdf.to_csv(csv_path, index=False)
# print('已将yyhmusic_260122_3.bdf转为csv。')

In [5]:
# Step 2: ICA去除伪迹
import numpy as np
ica_filtered_data = []
# 只处理一个文件
raw = mne.io.read_raw_bdf(bdf_path, preload=True, verbose=False)
# 预处理：滤波（1-40 Hz）和重参考
raw.filter(1., 40., fir_design='firwin', verbose=False)
raw.set_eeg_reference('average', verbose=False)
# 预处理：陷波滤波（50 Hz）
raw.notch_filter(50., fir_design='firwin', verbose=False)
# 坏道检测：自动标记坏道
raw.interpolate_bads(verbose=False)
# 预处理：重采样（256 Hz）
raw.resample(256, npad='auto', verbose=False)
# ICA分解
ica = mne.preprocessing.ICA(n_components=15, random_state=97, max_iter='auto')
ica.fit(raw)
# 自动检测并去除眼动/心跳等典型伪迹
eog_picks = mne.pick_types(raw.info, eog=True)
ecg_picks = mne.pick_types(raw.info, ecg=True)
if len(eog_picks) > 0:
    ica.find_bads_eog(raw, verbose=False)
if len(ecg_picks) > 0:
    ica.find_bads_ecg(raw, verbose=False)
ica.apply(raw)
ica_filtered_data.append(raw)
print('ICA处理完成，已去除伪迹。')

C:\Users\18439\AppData\Local\Temp\ipykernel_20056\3812317529.py:12: RuntimeWarning: No bad channels to interpolate. Doing nothing...
  raw.interpolate_bads(verbose=False)


Fitting ICA to data using 64 channels (please be patient, this may take a while)
Selecting by number: 15 components
Fitting ICA took 1.4s.
Applying ICA to Raw instance
    Transforming to ICA space (15 components)
    Zeroing out 0 ICA components
    Projecting back using 64 PCA components
ICA处理完成，已去除伪迹。


In [32]:
# Step 3: 滑窗特征提取+用music_timestamps.csv打标（EEG增强特征 v2）
import numpy as np
import pandas as pd
from scipy.signal import welch, detrend
from scipy.stats import skew, kurtosis

window_sec = 2
step_sec = 1
feature_list = []
label_list = []

# 读取music_timestamps.csv
ts_df = pd.read_csv(r'd:/projects/music_eeg/data_collector/save/music_timestamps.csv')

# 仅提取EEG通道，避免触发/状态通道污染特征
eeg_picks = mne.pick_types(raw.info, eeg=True, exclude='bads')
if len(eeg_picks) == 0:
    raise ValueError('未找到EEG通道，请检查数据通道类型。')

sfreq = raw.info['sfreq']
data = raw.get_data(picks=eeg_picks)
n_samples = data.shape[1]
window_size = int(window_sec * sfreq)
step_size = int(step_sec * sfreq)
n_windows = (n_samples - window_size) // step_size + 1

# 只保留音乐播放区间
music_rows = ts_df[ts_df['Event'].str.contains('音乐播放开始|音乐播放结束')].reset_index(drop=True)
music_blocks = []
for i in range(0, len(music_rows), 2):
    if i + 1 < len(music_rows):
        start = music_rows.loc[i, 'Unix_Timestamp']
        end = music_rows.loc[i + 1, 'Unix_Timestamp']
        label = music_rows.loc[i, 'Music_Type']
        music_blocks.append({'start': start, 'end': end, 'label': label})

# 频带定义（Hz）
bands = [
    ('delta', 1, 4),
    ('theta', 4, 8),
    ('alpha', 8, 13),
    ('beta', 13, 30),
    ('gamma', 30, 40),
]

def hjorth_parameters(x):
    x = detrend(x, axis=1, type='linear')
    dx = np.diff(x, axis=1)
    ddx = np.diff(dx, axis=1)
    var_x = np.var(x, axis=1) + 1e-12
    var_dx = np.var(dx, axis=1) + 1e-12
    var_ddx = np.var(ddx, axis=1) + 1e-12
    activity = var_x
    mobility = np.sqrt(var_dx / var_x)
    complexity = np.sqrt(var_ddx / var_dx) / (mobility + 1e-12)
    return activity, mobility, complexity

# 滑窗特征提取与打标
start_time = ts_df.loc[0, 'Unix_Timestamp']  # 实验开始时间
for i in range(n_windows):
    start = i * step_size
    end = start + window_size
    window_data = data[:, start:end]  # [n_channels, window_samples]

    # 时域基础特征
    mean_feat = window_data.mean(axis=1)
    std_feat = window_data.std(axis=1)
    rms_feat = np.sqrt((window_data ** 2).mean(axis=1))
    skew_feat = skew(window_data, axis=1, bias=False, nan_policy='omit')
    kurt_feat = kurtosis(window_data, axis=1, fisher=True, bias=False, nan_policy='omit')
    zcr_feat = np.mean(np.abs(np.diff(np.signbit(window_data), axis=1)), axis=1).astype(np.float32)

    # Hjorth参数
    hj_activity, hj_mobility, hj_complexity = hjorth_parameters(window_data)

    # 频域特征：相对带功率 + 谱熵 + 对数带功率
    freqs, psd = welch(window_data, fs=sfreq, nperseg=min(window_data.shape[1], 256), axis=1)
    total_power = psd.sum(axis=1, keepdims=True) + 1e-12
    psd_norm = psd / total_power
    spectral_entropy = -np.sum(psd_norm * np.log(psd_norm + 1e-12), axis=1)

    band_feats = []
    band_power_list = []
    for _, f_low, f_high in bands:
        idx = (freqs >= f_low) & (freqs < f_high)
        band_power = psd[:, idx].sum(axis=1) + 1e-12
        rel_power = band_power / total_power.ravel()
        band_feats.append(rel_power)
        band_power_list.append(band_power)

    # 常用带间比值（全通道均值后）
    delta_p = band_power_list[0].mean()
    theta_p = band_power_list[1].mean()
    alpha_p = band_power_list[2].mean()
    beta_p = band_power_list[3].mean()
    gamma_p = band_power_list[4].mean()
    ratio_feats = np.array([
        theta_p / (alpha_p + 1e-12),
        (theta_p + alpha_p) / (beta_p + 1e-12),
        beta_p / (alpha_p + 1e-12),
        gamma_p / (beta_p + 1e-12),
    ], dtype=np.float32)

    # 跨通道相关特征（上三角统计）
    corr_mat = np.corrcoef(window_data)
    corr_mat = np.nan_to_num(corr_mat, nan=0.0, posinf=0.0, neginf=0.0)
    tri_idx = np.triu_indices_from(corr_mat, k=1)
    corr_vals = corr_mat[tri_idx]
    if corr_vals.size == 0:
        corr_stats = np.array([0.0, 0.0, 0.0], dtype=np.float32)
    else:
        corr_stats = np.array([corr_vals.mean(), corr_vals.std(), np.median(corr_vals)], dtype=np.float32)

    # 协方差谱特征（前8个特征值）
    cov_mat = np.cov(window_data)
    eigvals = np.linalg.eigvalsh(cov_mat)
    eigvals = np.sort(np.clip(eigvals, 1e-12, None))[::-1]
    top_k = 8
    if eigvals.size < top_k:
        eigvals = np.pad(eigvals, (0, top_k - eigvals.size), mode='constant', constant_values=1e-12)
    eig_feat = np.log(eigvals[:top_k] + 1e-12).astype(np.float32)

    features = np.concatenate([
        mean_feat, std_feat, rms_feat, skew_feat, kurt_feat, zcr_feat,
        hj_activity, hj_mobility, hj_complexity,
        spectral_entropy,
        *band_feats,
        ratio_feats,
        corr_stats,
        eig_feat,
    ])
    features = np.nan_to_num(features, nan=0.0, posinf=0.0, neginf=0.0)
    feature_list.append(features.astype(np.float32))

    # 计算当前窗口的时间戳并打标
    window_start_time = start_time + start / sfreq
    label = 'unknown'
    for block in music_blocks:
        if block['start'] <= window_start_time < block['end']:
            label = block['label']
            break
    label_list.append(label)

feature_array = np.array(feature_list, dtype=np.float32)
df = pd.DataFrame(feature_array)
df['label'] = label_list
print('标签分布:')
print(df['label'].value_counts())
print(f'特征维度: {feature_array.shape[1]}')
df.head()

标签分布:
label
unknown      104
rock          35
ambient       34
classical     32
jazz          31
Name: count, dtype: int64
特征维度: 975


,0,1,2,3,4,5,6,7,8,9,...,966,967,968,969,970,971,972,973,974,label
0,-2.310093e-06,-2.043971e-06,2.919107e-07,8.390488e-07,1.071114e-06,6.662681e-07,1.034346e-07,-1.450925e-08,-8.743656e-08,6.318615e-07,...,-0.044770,-20.163837,-21.876930,-21.936218,-22.556063,-23.049961,-23.190138,-23.301085,-23.575918,unknown
1,1.035320e-06,4.670548e-07,9.708459e-07,2.379298e-07,3.652883e-07,6.116993e-09,1.188349e-07,3.247488e-07,-1.776381e-07,2.832924e-07,...,-0.061135,-21.254644,-21.790005,-22.231861,-22.489658,-23.036028,-23.174677,-23.409533,-23.785728,unknown
2,3.207251e-07,4.581173e-08,1.824511e-07,4.921310e-07,1.351832e-07,-4.629928e-08,1.123179e-07,1.333451e-06,8.865364e-07,-5.303032e-08,...,-0.043152,-21.072849,-21.465815,-21.938364,-22.384161,-23.030640,-23.217510,-23.318216,-23.763605,unknown
3,-1.555010e-06,-8.515572e-07,-7.497921e-07,-1.893097e-09,-1.446703e-07,-3.138640e-07,9.187520e-09,-7.124041e-07,5.889442e-07,-1.429246e-07,...,-0.027150,-20.569387,-21.330610,-22.365601,-22.447598,-22.799402,-23.227249,-23.562042,-23.867743,rock
4,-4.821740e-07,-7.512392e-07,-3.604995e-08,-4.110636e-07,-4.825504e-07,-8.193060e-07,-9.097488e-07,-1.177831e-07,-3.324423e-07,-2.316128e-08,...,-0.010959,-20.112707,-21.071318,-22.193050,-22.354408,-22.694635,-23.075848,-23.453537,-23.602661,rock


In [33]:
# Step 4: 剔除unknown、先划分再标准化、数据集保存
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1) 剔除无效标签
df_valid = df[df['label'] != 'unknown'].copy()
if df_valid.empty:
    raise ValueError("过滤unknown后没有可用样本，请检查打标逻辑或时间戳。")

print('过滤后标签分布:')
print(df_valid['label'].value_counts())

# 2) 先划分，避免数据泄漏
X = df_valid.drop('label', axis=1).values
y = df_valid['label'].values
X_train_raw, X_temp_raw, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
X_val_raw, X_test_raw, y_val, y_test = train_test_split(
    X_temp_raw, y_temp, test_size=1/3, random_state=42, stratify=y_temp
)

# 3) 仅用训练集拟合标准化器，再变换验证/测试集
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_val = scaler.transform(X_val_raw)
X_test = scaler.transform(X_test_raw)

# 4) 转存为csv
train_df = pd.DataFrame(X_train)
train_df['label'] = y_train
val_df = pd.DataFrame(X_val)
val_df['label'] = y_val
test_df = pd.DataFrame(X_test)
test_df['label'] = y_test

train_df.to_csv('train_features.csv', index=False)
val_df.to_csv('val_features.csv', index=False)
test_df.to_csv('test_features.csv', index=False)

print('已剔除unknown并完成无泄漏标准化、划分与转存。')
print(f'train/val/test: {len(train_df)}/{len(val_df)}/{len(test_df)}')

过滤后标签分布:
label
rock         35
ambient      34
classical    32
jazz         31
Name: count, dtype: int64
已剔除unknown并完成无泄漏标准化、划分与转存。
train/val/test: 92/26/14


# 说明
本 notebook 仅保留数据预处理与特征工程（Step 1-4）。
模型训练与评估已拆分到 `model_training.ipynb`。